# Factor Diagnostics Validation v1

本 notebook 验证 Alphalens 风格因子分析只能作为 `src/factor` 的 research hypothesis 和 license/provenance 参考，不复制 upstream 代码；可运行诊断使用本仓库 `src.factor` 稳定接口。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'AGENTS.md').exists() and (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.factor import analyze_factor_suite, build_composite_factor, compute_factor_ic

CARD = REPO_ROOT / 'research' / 'factor_diagnostics_validation_v1'
CARD.mkdir(parents=True, exist_ok=True)

SOURCE = {
    'source_id': 'github:quantopian/alphalens',
    'source_type': 'github',
    'title': 'Alphalens factor performance analysis',
    'url': 'https://github.com/quantopian/alphalens',
    'summary': (
        'GitHub factor-analysis reference for returns analysis, information coefficient analysis, '
        'turnover analysis, and grouped analysis; useful only as a src.factor diagnostics hypothesis.'
    ),
    'retrieved_at': '2026-06-19',
    'access_status': 'repository_accessible',
}
EVIDENCE_REQUIREMENTS = [
    'license_review',
    'research_notebook',
    'focused_pytest',
    'manifest_or_export',
    'chinese_conclusion',
]
SOURCE

{'source_id': 'github:quantopian/alphalens',
 'source_type': 'github',
 'title': 'Alphalens factor performance analysis',
 'url': 'https://github.com/quantopian/alphalens',
 'summary': 'GitHub factor-analysis reference for returns analysis, information coefficient analysis, turnover analysis, and grouped analysis; useful only as a src.factor diagnostics hypothesis.',
 'retrieved_at': '2026-06-19',
 'access_status': 'repository_accessible'}

In [2]:
price = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(price, horizon=1)
factor_columns = list(dataset.feature_columns)

report = analyze_factor_suite(
    dataset.frame,
    factor_columns=factor_columns,
    target_column=dataset.target_column,
    quantiles=5,
)
report.ic.to_csv(CARD / 'factor_ic.csv', index=False)
report.correlation.to_csv(CARD / 'factor_correlation.csv', index=False)
report.quantile_returns.to_csv(CARD / 'factor_quantile_returns.csv', index=False)

composite_weights = {
    str(row.factor): float(row.ic_mean)
    for row in report.ic.itertuples(index=False)
    if abs(float(row.ic_mean)) > 0.05
}
composite = build_composite_factor(
    dataset.frame,
    factor_columns=factor_columns,
    weights=composite_weights,
    output_column='ic_weighted_composite',
)
composite_ic = compute_factor_ic(
    composite,
    factor_columns=['ic_weighted_composite'],
    target_column=dataset.target_column,
)
composite[['date', 'ic_weighted_composite', dataset.target_column]].to_csv(
    CARD / 'composite_factor.csv', index=False
)
composite_ic.to_csv(CARD / 'composite_factor_ic.csv', index=False)

analysis_summary = {
    **report.summary,
    'start': str(dataset.frame['date'].min().date()),
    'end': str(dataset.frame['date'].max().date()),
    'row_count': int(len(dataset.frame)),
    'source_reference': SOURCE['source_id'],
    'diagnostic_interface': 'src.factor.analyze_factor_suite',
    'composite_factor': 'ic_weighted_composite',
    'composite_ic_mean': float(composite_ic.iloc[0]['ic_mean']),
    'artifact_files': [
        'factor_ic.csv',
        'factor_correlation.csv',
        'factor_quantile_returns.csv',
        'composite_factor.csv',
        'composite_factor_ic.csv',
    ],
}
(CARD / 'analysis_summary.json').write_text(
    json.dumps(analysis_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
report.ic.head(10)

,factor,scope,method,ic_mean,ic_std,ic_positive_rate,ic_negative_rate,observation_count,period_count
0,vix_change_1d,time_series,spearman,0.124268,0.0,1.0,0.0,108,1
1,realized_vol_21,time_series,spearman,0.088324,0.0,1.0,0.0,108,1
2,drawdown_21,time_series,spearman,0.074311,0.0,1.0,0.0,108,1
3,vix_level,time_series,spearman,0.060343,0.0,1.0,0.0,108,1
4,realized_vol_5,time_series,spearman,0.058488,0.0,1.0,0.0,108,1
5,intraday_range,time_series,spearman,0.051845,0.0,1.0,0.0,108,1
6,volume_z_20,time_series,spearman,0.044401,0.0,1.0,0.0,108,1
7,trend_21,time_series,spearman,0.039963,0.0,1.0,0.0,108,1
8,soxl_return_5d,time_series,spearman,0.034336,0.0,1.0,0.0,108,1
9,soxl_soxx_relative_5d,time_series,spearman,0.034041,0.0,1.0,0.0,108,1


In [3]:
source_provenance = {
    'schema_version': 'knowledge.source_provenance.v1',
    **SOURCE,
    'source_used_as': 'hypothesis_only',
    'external_source_checks': {
        'repository_url': 'https://github.com/quantopian/alphalens',
        'license_url': 'https://github.com/quantopian/alphalens/blob/master/LICENSE',
        'repository_license': 'Apache-2.0',
        'copied_external_code': False,
        'vendored_external_code': False,
    },
    'license_note': (
        '只保存 Alphalens GitHub URL、自有摘要和验证假设；未复制 README/代码正文；'
        '未 vendoring 第三方实现代码。'
    ),
}
(CARD / 'source_provenance.json').write_text(
    json.dumps(source_provenance, ensure_ascii=False, indent=2), encoding='utf-8'
)

license_review = """# License Review

## 中文结论

- GitHub 来源：https://github.com/quantopian/alphalens，仓库 license 复核为 Apache-2.0。
- 只保存 URL、自有摘要、结构化假设和本仓库运行产物；未复制 README 正文，未 vendoring Alphalens 或第三方实现代码。
- Alphalens 的 returns analysis、information coefficient analysis、turnover analysis、grouped analysis 只作为因子诊断方法假设。
- 本 notebook 只验证本仓库已有 `src.factor.analyze_factor_suite`、`compute_factor_ic`、`compute_factor_correlation` 和分层收益输出契约。
- 后续如要直接依赖 Alphalens runtime，需要单独复核依赖版本、维护状态和 Apache-2.0 notice 要求。
"""
(CARD / 'license_review.md').write_text(license_review, encoding='utf-8')

factor_summary = {
    'schema_version': 'research.factor_diagnostics_validation.v1',
    'source_id': SOURCE['source_id'],
    'candidate_name': 'factor_diagnostics',
    'target_subsystem': 'src/factor',
    'factor_report_schema_version': analysis_summary['schema_version'],
    'diagnostic_interface': 'src.factor.analyze_factor_suite',
    'top_factor': analysis_summary['top_factor'],
    'top_ic_mean': analysis_summary['top_ic_mean'],
    'composite_ic_mean': analysis_summary['composite_ic_mean'],
    'high_correlation_pair_count': analysis_summary['high_correlation_pair_count'],
    'artifact_files': analysis_summary['artifact_files'],
    'source_used_as': 'hypothesis_only',
    'implementation_allowed': False,
    'promotion_target': 'research_only',
    'evidence_requirements': EVIDENCE_REQUIREMENTS,
}
(CARD / 'factor_diagnostics_summary.json').write_text(
    json.dumps(factor_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)

artifact_manifest = {
    'schema_version': 'research.artifact_manifest.v1',
    'card_id': 'factor_diagnostics_validation_v1',
    'primary_notebook': 'factor_diagnostics_validation_v1.ipynb',
    'source_id': SOURCE['source_id'],
    'artifacts': factor_summary['artifact_files'] + [
        'analysis_summary.json',
        'factor_diagnostics_summary.json',
        'source_provenance.json',
        'license_review.md',
    ],
}
(CARD / 'artifact_manifest.json').write_text(
    json.dumps(artifact_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
factor_summary

{'schema_version': 'research.factor_diagnostics_validation.v1',
 'source_id': 'github:quantopian/alphalens',
 'candidate_name': 'factor_diagnostics',
 'target_subsystem': 'src/factor',
 'factor_report_schema_version': 'factor_suite_report.v1',
 'diagnostic_interface': 'src.factor.analyze_factor_suite',
 'top_factor': 'vix_change_1d',
 'top_ic_mean': 0.12426792293785408,
 'composite_ic_mean': 0.03079062943591795,
 'high_correlation_pair_count': 8,
 'artifact_files': ['factor_ic.csv',
  'factor_correlation.csv',
  'factor_quantile_returns.csv',
  'composite_factor.csv',
  'composite_factor_ic.csv'],
 'source_used_as': 'hypothesis_only',
 'implementation_allowed': False,
 'promotion_target': 'research_only',
 'evidence_requirements': ['license_review',
  'research_notebook',
  'focused_pytest',
  'manifest_or_export',
  'chinese_conclusion']}

## 中文结论

- 做了什么：将 Alphalens 作为外部因子诊断参考做 license/provenance 门禁，并用本仓库 `src.factor.analyze_factor_suite` 跑 SOXL 候选因子的 IC、相关性、分层收益和 IC 加权组合。
- 核心指标：本次评估 14 个候选因子，`analysis_summary.json` 记录 top factor、top IC、high correlation pair count、composite IC 和 artifact 文件列表。
- regime 差异：本 notebook 只验证 factor diagnostics 门禁，不新增 regime 结论；后续 regime 分层应走 `src.analysis` 的区间和 regime report。
- 是否晋升：否。当前只允许作为 research evidence，让 `factor_diagnostics` 通过外部知识门禁；不直接进入 production，不复制 Alphalens 代码。
- 风险：当前是 SOXL 单资产时间序列 IC，不能替代多资产横截面检验；高相关因子需要在多因子组合前去冗余或降权。
- 下一步：把通过 IC、相关性和分层收益筛选的因子接入模型矩阵，对比单因子、多因子、模型特征集和不同区间下的回测差异。